## 1. 시뮬레이션 
<핵심 가정>
- 차량 도착: 03:00 ± 15분
- 상차시간: 8~18분 (triangular)
- 주행속도: 평균 25km/h + 변동
- 서비스시간: 주문당 1~2.5분
- 수요: Poisson
- 운영시간: 240분 (03~07)

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [7]:
vrp_routes_df = pd.read_csv('data/vrp_routes.csv')
vrp_routes_df.head()

,candidate_index,candidate_name,vehicle_id,route_nodes,visited_demand_indices,route_load,route_distance_km
0,0,상명대학교사범대학부속여자고등학교,0,"[0, 23, 19, 15, 10, 9, 8, 20, 21, 14, 4, 16, 3...","[2009, 1757, 1238, 778, 716, 694, 1851, 1886, ...",24,7.783
1,0,상명대학교사범대학부속여자고등학교,1,"[0, 32, 26, 7, 18, 5, 30, 6, 17, 13, 22, 1, 2,...","[2770, 2480, 633, 1555, 327, 2622, 415, 1317, ...",21,7.864
2,1,대원국제중학교,0,"[0, 9, 47, 40, 8, 12, 22, 27, 0]","[844, 2747, 2348, 746, 1276, 1712, 1906]",11,23.609
3,1,대원국제중학교,1,"[0, 16, 34, 11, 13, 7, 28, 20, 44, 1, 0]","[1428, 2305, 1104, 1350, 605, 1952, 1517, 2430...",24,24.906
4,1,대원국제중학교,2,"[0, 35, 25, 4, 18, 32, 45, 42, 2, 41, 31, 3, 0]","[2306, 1872, 419, 1452, 2204, 2542, 2380, 296,...",24,25.419


1) 단일 시뮬레이션

In [8]:
import numpy as np
import pandas as pd


def simulate_one_day(
    vrp_routes_df,
    avg_speed_kmh=25,
    horizon_min=240
):
    results = []

    # MFC별로 묶기 (상차 대기열 계산 위해)
    for mfc, group in vrp_routes_df.groupby("candidate_index"):

        # 차량 도착시간 (03:00 기준)
        arrivals = np.random.normal(loc=0, scale=5, size=len(group))

        # FIFO 정렬
        order = np.argsort(arrivals)
        current_time = 0  # 상차 서버 1개 기준

        for idx in order:
            row = group.iloc[idx]

            # ===== 1. 수요 랜덤 발생 =====
            # 기존 route_load를 평균으로 사용
            demand = np.random.poisson(max(row["route_load"], 1))

            # ===== 2. 상차 =====
            arrival = arrivals[idx]

            start_loading = max(arrival, current_time)

            loading_time = np.random.triangular(8, 12, 18)

            end_loading = start_loading + loading_time

            current_time = end_loading

            # ===== 3. 주행 =====
            distance = row["route_distance_km"]

            drive_time = (distance / avg_speed_kmh) * 60
            drive_time *= np.random.lognormal(mean=0, sigma=0.1)

            # ===== 4. 배송 =====
            service_time = demand * np.random.triangular(1, 1.5, 2.5)

            # ===== 총 시간 =====
            total_time = end_loading + drive_time + service_time

            results.append({
                "candidate_index": mfc,
                "vehicle_id": row["vehicle_id"],
                "finish_time": total_time,
                "on_time": total_time <= horizon_min
            })

    return pd.DataFrame(results)

2. 몬테카를로 반복

In [9]:
def monte_carlo_simulation(
    vrp_routes_df,
    n_sim=300
):
    all_results = []

    for i in range(n_sim):
        sim_df = simulate_one_day(vrp_routes_df)
        sim_df["sim"] = i
        all_results.append(sim_df)

    result = pd.concat(all_results, ignore_index=True)

    return result

3) 실행

In [10]:
sim_result = monte_carlo_simulation(
    vrp_routes_df,
    n_sim=300
)

4) 결과 분석

4-1) 전체 완료율

In [11]:
overall_rate = sim_result["on_time"].mean()
print("전체 7시 전 완료율:", overall_rate)

전체 7시 전 완료율: 0.8594141414141414


4-2) MFC별 완료율

In [12]:
mfc_perf = (
    sim_result.groupby("candidate_index")["on_time"]
    .mean()
    .reset_index()
    .rename(columns={"on_time": "on_time_rate"})
)

print(mfc_perf.head())

   candidate_index  on_time_rate
0                0      1.000000
1                1      1.000000
2                2      0.996364
3               10      0.966333
4               15      0.497879


4-3) Worst Senario

In [13]:
worst = sim_result.groupby("sim")["on_time"].mean().min()
print("최악 시나리오 완료율:", worst)

최악 시나리오 완료율: 0.8121212121212121


4-4) 지연 차량 확인

In [16]:
late_cases = sim_result[sim_result["on_time"] == False]
print("지연 발생 수:", len(late_cases))
# ratio 출력
late_ratio = len(late_cases) / len(sim_result)
print("지연 발생 비율:", late_ratio)

지연 발생 수: 6959
지연 발생 비율: 0.1405858585858586
